In [10]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

#### Show Parquet as DF

In [5]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\PV.EST\PV.EST.parquet")
df

,country,year,value
0,Aruba,1960,NaN
1,Aruba,1961,NaN
2,Aruba,1962,NaN
3,Aruba,1963,NaN
4,Aruba,1964,NaN
...,...,...,...
17019,Zimbabwe,2019,-0.943303
17020,Zimbabwe,2020,-1.052743
17021,Zimbabwe,2021,-0.954443
17022,Zimbabwe,2022,-0.894974


#### Shrinking

In [ ]:
output_path = r"/workspace/qgi\qgi-4.0\data\etl_output\PV.EST\PV.EST_shrinked.parquet"

# Select 5 countries to keep
target_countries = ["United States", "China", "India", "Japan", "Germany"]

# Filter the DataFrame
filtered_df = df[df["country"].isin(target_countries)]
filtered_df

,country,year,value
2560,China,1960,NaN
2561,China,1961,NaN
2562,China,1962,NaN
2563,China,1963,NaN
2564,China,1964,NaN
...,...,...,...
16123,United States,2019,0.118299
16124,United States,2020,-0.027900
16125,United States,2021,-0.014425
16126,United States,2022,0.008701


In [ ]:
filtered_df.to_parquet(output_path, index = False, compression = "snappy")
print(f"Shrinked Parquet file saved to {output_path}")

Shrinked Parquet file saved to /workspace/qgi\qgi-4.0\data\etl_output\PV.EST\PV.EST_shrinked.parquet


#### Calculating Comaprisons

In [11]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\etl_output\PV.EST_shrinked\PV.EST_shrinked.parquet")


def preprocess_data(df):
    """
    Preprocesses data for each country by handling NaN values and organizing data for analysis.
    
    This function processes the input DataFrame to prepare data for each country:
    1. Filters the data for each country
    2. Sorts the data by year
    3. Handles NaN values by either skipping countries with more than one NaN or interpolating single NaN values
    4. Creates a list of dictionaries containing the country name, original data, and cleaned values
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    
    Returns:
    list: A list of dictionaries, where each dictionary contains:
        - "country": The country name
        - "data": The original country data sorted by year
        - "values": Cleaned numeric values with NaNs handled
    """
    countries = df["country"].unique()
    preprocessed_data = []

    for country in countries:
        country_data = df[df["country"] == country].sort_values("year")
        values = country_data["value"].values.copy()

        # Handle NaN values
        nan_indices = np.isnan(values)
        if np.sum(nan_indices) > 1:
            continue

        if np.sum(nan_indices) == 1:
            nan_pos = np.where(nan_indices)[0][0]
            if nan_pos == 0:
                values[nan_pos] = values[nan_pos + 1]
            elif nan_pos == len(values) - 1:
                values[nan_pos] = values[nan_pos - 1]
            else:
                values[nan_pos] = (values[nan_pos - 1] + values[nan_pos + 1]) / 2

        preprocessed_data.append({
            "country": country,
            "data": country_data,
            "values": values
        })

    return preprocessed_data

def calculate_total_comparisons(df, min_length=3, max_length=5):
    """
    Calculate the total number of segment comparisons for progress tracking.
    
    This function computes the total number of comparisons that would be made 
    during the pattern extraction process. It considers all possible segment 
    lengths within the specified range and all country pairs.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    min_length (int): Minimum segment length to consider
    max_length (int): Maximum segment length to consider
    
    Returns:
    int: Total number of comparisons that would be performed
    """
    preprocessed_data = preprocess_data(df)
    total_comparisons = 0

    for i, country_a in enumerate(preprocessed_data):
        values_a = country_a["values"]

        for segment_length in range(min_length, max_length + 1):
            if len(values_a) < segment_length:
                continue

            num_segments_a = len(values_a) - segment_length + 1
            num_segments_b = sum(
                len(country_b["values"]) - segment_length + 1
                for country_b in preprocessed_data[i+1:]
                if len(country_b["values"]) >= segment_length
            )
            total_comparisons += num_segments_a * num_segments_b

    return total_comparisons

# Example usage:
total = calculate_total_comparisons(df)
print(f"Total comparisons: {total}")

Total comparisons: 0


#### Size Saving Comparison

In [14]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\patternizer_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked_sdcis.parquet")
print(df.head())

   country_a_id  country_b_id  length  start_year_a  start_year_b  correlation
0            45            81       3          1960          1977     0.945475
1            45            81       3          1961          1960     0.970550
2            45            81       3          1961          1961     0.964687
3            45            81       3          1961          1962     0.948679
4            45            81       3          1961          1963     0.965124


In [15]:
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,45,81,3,1960,1977,0.945475
1,45,81,3,1961,1960,0.970550
2,45,81,3,1961,1961,0.964687
3,45,81,3,1961,1962,0.948679
4,45,81,3,1961,1963,0.965124
...,...,...,...,...,...,...
82764,110,236,5,2005,2015,0.958875
82765,110,236,5,2005,2017,0.943587
82766,110,236,5,2005,2018,0.978710
82767,110,236,5,2005,2019,0.901636


In [13]:
df

,country_a,country_b,length,start_year_a,start_year_b,correlation,data_segment_a,data_segment_b
0,China,Germany,3,1960,1977,0.945475,"[667070000.0, 660330000.0, 665770000.0]","[78159814.0, 78091820.0, 78126350.0]"
1,China,Germany,3,1961,1960,0.970550,"[660330000.0, 665770000.0, 682335000.0]","[72814900.0, 73377632.0, 74025784.0]"
2,China,Germany,3,1961,1961,0.964687,"[660330000.0, 665770000.0, 682335000.0]","[73377632.0, 74025784.0, 74714353.0]"
3,China,Germany,3,1961,1962,0.948679,"[660330000.0, 665770000.0, 682335000.0]","[74025784.0, 74714353.0, 75318337.0]"
4,China,Germany,3,1961,1963,0.965124,"[660330000.0, 665770000.0, 682335000.0]","[74714353.0, 75318337.0, 75963695.0]"
...,...,...,...,...,...,...,...,...
82764,Japan,United States,5,2005,2015,0.958875,"[127773000.0, 127854000.0, 128001000.0, 128063...","[320738994.0, 323071755.0, 325122128.0, 326838..."
82765,Japan,United States,5,2005,2017,0.943587,"[127773000.0, 127854000.0, 128001000.0, 128063...","[325122128.0, 326838199.0, 328329953.0, 331526..."
82766,Japan,United States,5,2005,2018,0.978710,"[127773000.0, 127854000.0, 128001000.0, 128063...","[326838199.0, 328329953.0, 331526933.0, 332048..."
82767,Japan,United States,5,2005,2019,0.901636,"[127773000.0, 127854000.0, 128001000.0, 128063...","[328329953.0, 331526933.0, 332048977.0, 333271..."
